# Primitive Motion Tests — SO-101

Test each motion primitive independently. Reset to WORK_POSITION between each test.

**WORK_POSITION**: shoulder_pan=-18, shoulder_lift=-60, elbow_flex=37, wrist_flex=74, wrist_roll=0, gripper=0

In [2]:
from lerobot.model.kinematics import RobotKinematics

In [3]:
kinematics = RobotKinematics(urdf_path="mcp_server/sim/assets/so101/so101.urdf")

  -base_link_2 collides with shoulder_link_2
  -shoulder_link_0 collides with upper_arm_link_1
  -upper_arm_link_0 collides with lower_arm_link_0
  -lower_arm_link_2 collides with wrist_link_1
  -wrist_link_0 collides with gripper_link_1
  -gripper_link_0 collides with moving_jaw_so101_v1_link_0


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))

os.environ.setdefault("SIMULATE", "false")
os.environ.setdefault("LEROBOT_FOLLOWER_PORT", "/dev/ttyACM0")

from mcp_server.robot.kinematics import joints_to_cartesian, cartesian_to_joints
from mcp_server.robot.lerobot import LeRobotInterface

# ── Delta to test ─────────────────────────────────────────────────────────────
delta_x = 0.0
delta_y = 0.0
delta_z = 0.05

# ── Connect and read live state ───────────────────────────────────────────────
robot = LeRobotInterface(port="/dev/ttyACM0")

current_joints = robot.get_joint_positions()
current_ee     = robot.get_ee_position()

print("=== LIVE HARDWARE STATE (before move) ===")
print(f"  joints (hw°): {current_joints}")
print(f"  EE FK  (m):   x={current_ee[0]:.4f}  y={current_ee[1]:.4f}  z={current_ee[2]:.4f}")

# ── Plan: same computation the tool will run ──────────────────────────────────
target_ee = [current_ee[0] + delta_x, current_ee[1] + delta_y, current_ee[2] + delta_z]

print(f"\n=== PLANNED TARGET ===")
print(f"  delta  (m):   dx={delta_x}  dy={delta_y}  dz={delta_z}")
print(f"  target (m):   x={target_ee[0]:.4f}  y={target_ee[1]:.4f}  z={target_ee[2]:.4f}")

# ── IK: seeded from LIVE joints (same as tool) ────────────────────────────────
ik_joints = cartesian_to_joints(target_ee[0], target_ee[1], target_ee[2],
                                 seed_hw_joints=current_joints)

print(f"\n=== IK RESULT (hw°) — seeded from live joints ===")
for name in ["shoulder_pan","shoulder_lift","elbow_flex","wrist_flex","wrist_roll"]:
    before = current_joints.get(name, 0)
    after  = ik_joints.get(name, 0)
    print(f"  {name:>15}: {before:+7.2f}° → {after:+7.2f}°   (Δ {after-before:+.2f}°)")

# ── IK round-trip: FK(IK_joints) — model-only error ─────────────────────────
ik_ee = joints_to_cartesian({**ik_joints, "gripper": current_joints.get("gripper", 0)})

print(f"\n=== IK ROUND-TRIP FK (model error, no hardware) ===")
print(f"  FK of IK joints: x={ik_ee[0]:.4f}  y={ik_ee[1]:.4f}  z={ik_ee[2]:.4f}")
print(f"  model error (m): dx={ik_ee[0]-target_ee[0]:.4f}  dy={ik_ee[1]-target_ee[1]:.4f}  dz={ik_ee[2]-target_ee[2]:.4f}")

# ── Execute move ──────────────────────────────────────────────────────────────
print(f"\n=== CALLING move_cartesian_delta(dx={delta_x}, dy={delta_y}, dz={delta_z}) ===")
result = robot.move_cartesian_delta(delta_x, delta_y, delta_z)
print(f"  raw return: {result}")

# ── Read post-move state ──────────────────────────────────────────────────────
post_joints = robot.get_joint_positions()
post_ee     = robot.get_ee_position()

print(f"\n=== LIVE HARDWARE STATE (after move) ===")
print(f"  joints (hw°): {post_joints}")
print(f"  EE FK  (m):   x={post_ee[0]:.4f}  y={post_ee[1]:.4f}  z={post_ee[2]:.4f}")

# ── Separate model error vs hardware tracking error ───────────────────────────
achieved = [post_ee[i] - current_ee[i] for i in range(3)]
model_err = [ik_ee[i] - target_ee[i] for i in range(3)]
hw_err    = [post_ee[i] - ik_ee[i] for i in range(3)]

print(f"\n=== ERROR BREAKDOWN ===")
print(f"  planned delta:        dx={delta_x:.4f}  dy={delta_y:.4f}  dz={delta_z:.4f}")
print(f"  achieved delta:       dx={achieved[0]:.4f}  dy={achieved[1]:.4f}  dz={achieved[2]:.4f}")
print(f"  -- model error  (IK→FK):         dx={model_err[0]:.4f}  dy={model_err[1]:.4f}  dz={model_err[2]:.4f}")
print(f"  -- hardware error (cmd→actual):  dx={hw_err[0]:.4f}  dy={hw_err[1]:.4f}  dz={hw_err[2]:.4f}")


## Sign Map Calibration
Move one joint at a time by +10°. Compare FK-predicted EE delta with physical motion.
If direction is **opposite**, that joint's sign needs to flip.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
os.environ.setdefault("SIMULATE", "false")
os.environ.setdefault("LEROBOT_FOLLOWER_PORT", "/dev/ttyACM0")

from mcp_server.robot.kinematics import joints_to_cartesian
from mcp_server.robot.lerobot import LeRobotInterface

WORK_HW = dict(shoulder_pan=-18, shoulder_lift=-60, elbow_flex=37, wrist_flex=74, wrist_roll=0, gripper=0)
DELTA_DEG = 10.0

base_ee = joints_to_cartesian(WORK_HW)

print(f"WORK_POSITION EE (m): x={base_ee[0]:.4f}  y={base_ee[1]:.4f}  z={base_ee[2]:.4f}")
print(f"\nFK predictions for each joint +{DELTA_DEG}° (hw):")
print(f"{'joint':>15}  {'FK dx':>8}  {'FK dy':>8}  {'FK dz':>8}   FK says EE moves...")
print("-" * 70)

JOINT_NAMES = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"]

fk_deltas = {}
for name in JOINT_NAMES:
    test = {**WORK_HW, name: WORK_HW[name] + DELTA_DEG}
    ee = joints_to_cartesian(test)
    dx, dy, dz = ee[0]-base_ee[0], ee[1]-base_ee[1], ee[2]-base_ee[2]
    fk_deltas[name] = (dx, dy, dz)

    desc = []
    if abs(dz) > 0.001: desc.append(f"{'UP' if dz > 0 else 'DOWN'} {abs(dz)*100:.1f}cm")
    if abs(dx) > 0.001: desc.append(f"{'FORWARD' if dx > 0 else 'BACKWARD'} {abs(dx)*100:.1f}cm")
    if abs(dy) > 0.001: desc.append(f"{'LEFT' if dy > 0 else 'RIGHT'} {abs(dy)*100:.1f}cm")
    if not desc: desc = ["(no significant movement)"]

    print(f"  {name:>15}  {dx:>+8.4f}  {dy:>+8.4f}  {dz:>+8.4f}   {', '.join(desc)}")


In [ ]:
# ── Physical test: one joint at a time ───────────────────────────────────────
# Run this cell once per joint. Change JOINT_TO_TEST each time.
# Watch the physical arm and note if direction matches FK above.

JOINT_TO_TEST = "shoulder_lift"   # <-- change this each run
# Options: "shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"

robot = LeRobotInterface(port="/dev/ttyACM0")

# Step 1: go to WORK_POSITION
print(f"[1] Sending to WORK_POSITION...")
robot.send_joint_positions(WORK_HW)

import time; time.sleep(1.5)

# Step 2: read actual position after settling
before_joints = robot.get_joint_positions()
before_ee = robot.get_ee_position()
print(f"    joints: {before_joints}")
print(f"    EE FK:  x={before_ee[0]:.4f}  y={before_ee[1]:.4f}  z={before_ee[2]:.4f}")

# Step 3: apply +DELTA_DEG to the test joint only
target = {**before_joints, JOINT_TO_TEST: before_joints[JOINT_TO_TEST] + DELTA_DEG}
print(f"\n[2] Moving {JOINT_TO_TEST}: {before_joints[JOINT_TO_TEST]:.2f}° → {target[JOINT_TO_TEST]:.2f}° (+{DELTA_DEG}°)")

dx_pred, dy_pred, dz_pred = fk_deltas[JOINT_TO_TEST]
print(f"    FK predicts: dx={dx_pred:+.4f}  dy={dy_pred:+.4f}  dz={dz_pred:+.4f}")
print(f"    >>> WATCH THE ARM <<<")

robot.send_joint_positions(target)
time.sleep(1.5)

# Step 4: read result
after_joints = robot.get_joint_positions()
after_ee = robot.get_ee_position()
achieved_dx = after_ee[0] - before_ee[0]
achieved_dy = after_ee[1] - before_ee[1]
achieved_dz = after_ee[2] - before_ee[2]

print(f"\n[3] After move:")
print(f"    joints: {after_joints}")
print(f"    EE FK:  x={after_ee[0]:.4f}  y={after_ee[1]:.4f}  z={after_ee[2]:.4f}")
print(f"    FK delta: dx={achieved_dx:+.4f}  dy={achieved_dy:+.4f}  dz={achieved_dz:+.4f}")
print(f"\n[4] Does physical motion DIRECTION match FK?")
print(f"    Z: FK={'UP' if dz_pred>0 else 'DOWN'}  — did it go UP or DOWN physically?")
print(f"    X: FK={'FORWARD' if dx_pred>0 else 'BACKWARD'}  — did it go FORWARD or BACKWARD physically?")
print(f"    Y: FK={'LEFT' if dy_pred>0 else 'RIGHT'}  — did it go LEFT or RIGHT physically?")


## Workspace Bounds Mapping
Compute FK at key configurations to determine correct workspace limits for config.py.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
from mcp_server.robot.kinematics import joints_to_cartesian

# Key reference configurations (hw degrees)
configs = {
    "REST":        dict(shoulder_pan=-18, shoulder_lift=-105, elbow_flex=97,  wrist_flex=74, wrist_roll=0, gripper=0),
    "WORK":        dict(shoulder_pan=-18, shoulder_lift=-60,  elbow_flex=37,  wrist_flex=74, wrist_roll=0, gripper=0),
    # Extremes — arm swept to find bounds
    "arm_high":    dict(shoulder_pan=-18, shoulder_lift=-10,  elbow_flex=10,  wrist_flex=74, wrist_roll=0, gripper=0),
    "arm_low":     dict(shoulder_pan=-18, shoulder_lift=-80,  elbow_flex=60,  wrist_flex=74, wrist_roll=0, gripper=0),
    "arm_far":     dict(shoulder_pan=-18, shoulder_lift=-45,  elbow_flex=10,  wrist_flex=74, wrist_roll=0, gripper=0),
    "arm_close":   dict(shoulder_pan=-18, shoulder_lift=-80,  elbow_flex=80,  wrist_flex=74, wrist_roll=0, gripper=0),
    "pan_left":    dict(shoulder_pan=40,  shoulder_lift=-60,  elbow_flex=37,  wrist_flex=74, wrist_roll=0, gripper=0),
    "pan_right":   dict(shoulder_pan=-76, shoulder_lift=-60,  elbow_flex=37,  wrist_flex=74, wrist_roll=0, gripper=0),
}

print(f"{'config':>12}   {'x':>7}  {'y':>7}  {'z':>7}   (meters)")
print("-" * 50)
xs, ys, zs = [], [], []
for name, joints in configs.items():
    ee = joints_to_cartesian(joints)
    xs.append(ee[0]); ys.append(ee[1]); zs.append(ee[2])
    print(f"  {name:>12}   x={ee[0]:+.3f}  y={ee[1]:+.3f}  z={ee[2]:+.3f}")

print(f"\n  x range: [{min(xs):.3f}, {max(xs):.3f}]")
print(f"  y range: [{min(ys):.3f}, {max(ys):.3f}]")
print(f"  z range: [{min(zs):.3f}, {max(zs):.3f}]")
print(f"\nCurrent config.py limits:")
print(f"  x: [0.0, 0.6]   y: [-0.3, 0.3]   z: [0.0, 0.5]")
print(f"\n⚠ Adjust config.py WORKSPACE to match actual reachable range above.")


In [ ]:
# ── Incremental limit finder ──────────────────────────────────────────────────
# Set DIRECTION and run repeatedly. Each run moves one step.
# When the arm reaches its limit (unsafe/strained), note the EE position shown.
# Run reset() to go back to WORK_POSITION at any time.

import os, sys, time
sys.path.insert(0, os.path.abspath('.'))
os.environ.setdefault("SIMULATE", "false")
os.environ.setdefault("LEROBOT_FOLLOWER_PORT", "/dev/ttyACM0")

from mcp_server.robot.kinematics import joints_to_cartesian
from mcp_server.robot.lerobot import LeRobotInterface

DIRECTION = "+z"   # options: "+x", "-x", "+y", "-y", "+z", "-z"
STEP_M    = 0.02   # 2 cm per run

_dir_map = {
    "+x": (1,0,0), "-x": (-1,0,0),
    "+y": (0,1,0), "-y": (0,-1,0),
    "+z": (0,0,1), "-z": (0,0,-1),
}
dx, dy, dz = [v * STEP_M for v in _dir_map[DIRECTION]]

robot = LeRobotInterface(port="/dev/ttyACM0")
result = robot.move_cartesian_delta(dx, dy, dz)

joints  = robot.get_joint_positions()
ee      = robot.get_ee_position()
print(f"  direction: {DIRECTION}  step: {STEP_M*100:.0f}cm")
print(f"  EE now:  x={ee[0]:.3f}  y={ee[1]:.3f}  z={ee[2]:.3f}")
print(f"  joints:  {joints}")
print(f"  status:  {result.get('status')}")

def reset():
    WORK_HW = dict(shoulder_pan=-18, shoulder_lift=-60, elbow_flex=37,
                   wrist_flex=74, wrist_roll=0, gripper=0)
    robot.send_joint_positions(WORK_HW)
    time.sleep(1.0)
    ee = robot.get_ee_position()
    print(f"  reset → EE: x={ee[0]:.3f}  y={ee[1]:.3f}  z={ee[2]:.3f}")


In [ ]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath('.'))

os.environ.setdefault("SIMULATE", "false")
os.environ.setdefault("LEROBOT_FOLLOWER_PORT", "/dev/ttyACM0")

from mcp_server import server

def mcp(tool: str, **kwargs) -> dict:
    """Call an MCP tool function directly and return parsed result."""
    fn = getattr(server, tool)
    raw = fn(**kwargs)
    return json.loads(raw)

WORK = dict(shoulder_pan=-18, shoulder_lift=-60, elbow_flex=37, wrist_flex=74, wrist_roll=0, gripper=0)

def reset():
    result = mcp('send_joints', **WORK)
    time.sleep(1.5)
    return result

print("MCP wired. Robot will connect on first tool call.")


## 1. Move Left (+Y)

In [ ]:
reset()
result = mcp('move_left', distance_m=0.05)
print(result)
# Expected: EE moves to robot's left (+Y increases)

## 2. Move Right (-Y)

In [ ]:
reset()
result = mcp('move_right', distance_m=0.05)
print(result)
# Expected: EE moves to robot's right (-Y decreases)

## 3. Move Up (+Z)

In [ ]:
reset()
result = mcp('move_up', distance_m=0.05)
print(result)
# Expected: EE rises (+Z increases)

## 4. Move Down (-Z)

In [ ]:
reset()
result = mcp('move_down', distance_m=0.05)
print(result)
# Expected: EE lowers (-Z decreases)

## 5. Move Forward (+X)

In [ ]:
reset()
result = mcp('move_forward', distance_m=0.05)
print(result)
# Expected: EE moves away from base (+X increases)

## 6. Move Backward (-X)

In [ ]:
reset()
result = mcp('move_back', distance_m=0.05)
print(result)
# Expected: EE moves toward base (-X decreases)

## 7. Rotate Gripper Clockwise (+degrees)

In [ ]:
reset()
result = mcp('rotate_gripper', degrees=30)
print(result)
# Expected: wrist_roll rotates clockwise

## 8. Rotate Gripper Counter-Clockwise (-degrees)

In [ ]:
reset()
result = mcp('rotate_gripper', degrees=-30)
print(result)
# Expected: wrist_roll rotates counter-clockwise

## 9. Tilt Gripper

In [ ]:
reset()
result = mcp('tilt_gripper', degrees=20)
print(result)
# Expected: wrist_flex tilts gripper tip

## 10. Grasp / Release

In [ ]:
reset()
result_grasp = mcp('grasp')
print('grasp:', result_grasp)
# Expected: gripper closes (0 = closed)

result_release = mcp('release')
print('release:', result_release)
# Expected: gripper opens (100 = open)